In [0]:
# Databricks notebook source

# COMMAND ----------

%pip install twilio

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------

from pyspark.sql import functions as F
from twilio.rest import Client


# ============================================================
# CONFIG
# ============================================================

GOLD_CURRENT_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "gold/current_wildfire_risk"
)

SUBSCRIBER_TABLE = (
    "dbw_sce_wildfire_dev.default.wildfire_alert_subscribers"
)

ALERT_RISK_CATEGORIES = [
    "High",
    "Critical"
]

RISK_CHANGE_THRESHOLD = 10

MAX_ALERT_ZONES = 5


# IMPORTANT:
# True  = Twilio trial test
# False = real wildfire alert logic
TEST_MODE = False


# ============================================================
# LOAD CURRENT GOLD DATA
# ============================================================

df = (
    spark.read
    .format("delta")
    .load(GOLD_CURRENT_PATH)
)


current_count = df.count()


print(
    "Current Gold rows:",
    current_count
)


# ============================================================
# IDENTIFY ALERT ZONES
# ============================================================

alert_df = (
    df
    .filter(
        (
            F.col("risk_category")
            .isin(ALERT_RISK_CATEGORIES)
        )
        |
        (
            F.col("risk_change")
            >= RISK_CHANGE_THRESHOLD
        )
        |
        (
            F.col("category_changed")
            == True
        )
    )
    .orderBy(
        F.desc("risk_score"),
        F.desc("risk_change")
    )
)


alert_count = alert_df.count()


print(
    "Alert zones:",
    alert_count
)


# ============================================================
# DISPLAY ALERT CANDIDATES
# ============================================================

display(
    alert_df.select(
        "zone_id",
        "zone_label",
        "location_name",
        "risk_score",
        "risk_category",
        "previous_risk_score",
        "previous_category",
        "risk_change",
        "wind_gust_mph",
        "relative_humidity_pct",
        "temperature_f",
        "google_maps_url",
        "pipeline_run_timestamp"
    )
)


# ============================================================
# LOAD ACTIVE SUBSCRIBERS
# ============================================================

subscribers = (
    spark.table(
        SUBSCRIBER_TABLE
    )
    .filter(
        F.col("active") == True
    )
    .select(
        "phone_number"
    )
    .where(
        F.col("phone_number").isNotNull()
    )
    .dropDuplicates()
    .collect()
)


subscriber_count = len(
    subscribers
)


print(
    "Active subscribers:",
    subscriber_count
)


# ============================================================
# EXIT IF NO SUBSCRIBERS
# ============================================================

if subscriber_count == 0:

    print(
        "No active phone subscribers."
    )

    dbutils.notebook.exit(
        "NO_SUBSCRIBERS"
    )


# ============================================================
# GET TWILIO SECRETS
# ============================================================

account_sid = dbutils.secrets.get(
    scope="wildfire-alerts",
    key="twilio-account-sid"
)


auth_token = dbutils.secrets.get(
    scope="wildfire-alerts",
    key="twilio-auth-token"
)


from_number = dbutils.secrets.get(
    scope="wildfire-alerts",
    key="twilio-from-number"
)


# ============================================================
# CREATE TWILIO CLIENT
# ============================================================

client = Client(
    account_sid,
    auth_token
)


# ============================================================
# CONTROLLED TWILIO TRIAL TEST
# ============================================================

if TEST_MODE:

    # Trial accounts only allow approved template names.
    test_body = "sms_internal_alerts"

    test_success_count = 0
    test_failure_count = 0


    for subscriber in subscribers:

        to_number = (
            subscriber[
                "phone_number"
            ]
            .strip()
        )


        try:

            message = client.messages.create(
                body=test_body,
                from_=from_number,
                to=to_number
            )


            test_success_count += 1


            print(
                "Test SMS sent successfully."
            )


            print(
                "Twilio SID:",
                message.sid
            )


            print(
                "Recipient ending:",
                to_number[-4:]
            )


        except Exception as err:

            test_failure_count += 1


            print(
                "Test SMS failed for recipient "
                f"ending {to_number[-4:]}"
            )


            print(
                "Twilio error:",
                str(err)
            )


    print(
        "--------------------------------"
    )

    print(
        "Test SMS successful:",
        test_success_count
    )

    print(
        "Test SMS failed:",
        test_failure_count
    )


    if test_success_count > 0:

        dbutils.notebook.exit(
            "TEST_SMS_SENT"
        )

    else:

        dbutils.notebook.exit(
            "TEST_SMS_FAILED"
        )


# ============================================================
# REAL ALERT EXIT
# ============================================================

if alert_count == 0:

    print(
        "No High/Critical, major risk-increase, "
        "or category-change alerts."
    )

    dbutils.notebook.exit(
        "NO_ALERTS"
    )


# ============================================================
# BUILD REAL ALERT MESSAGE
# ============================================================

alert_rows = (
    alert_df
    .limit(MAX_ALERT_ZONES)
    .collect()
)


message_lines = [
    "WESTERN LA WILDFIRE ALERT",
    "",
    f"{alert_count} zone(s) meet alert criteria.",
    ""
]


for row in alert_rows:

    zone_label = (
        row["zone_label"]
        if row["zone_label"]
        else row["zone_id"]
    )


    risk_score = (
        float(row["risk_score"])
        if row["risk_score"] is not None
        else 0
    )


    risk_change = (
        float(row["risk_change"])
        if row["risk_change"] is not None
        else 0
    )


    gust = (
        float(row["wind_gust_mph"])
        if row["wind_gust_mph"] is not None
        else 0
    )


    humidity = (
        float(
            row["relative_humidity_pct"]
        )
        if row[
            "relative_humidity_pct"
        ] is not None
        else 0
    )


    message_lines.append(
        zone_label
    )


    message_lines.append(
        f"Risk: {risk_score:.1f} "
        f"({row['risk_category']})"
    )


    message_lines.append(
        f"Change: {risk_change:+.1f}"
    )


    message_lines.append(
        f"Gust: {gust:.1f} mph | "
        f"RH: {humidity:.0f}%"
    )


    if row["google_maps_url"]:

        message_lines.append(
            row["google_maps_url"]
        )


    message_lines.append("")


# ============================================================
# ADD REMAINING ALERT COUNT
# ============================================================

if alert_count > MAX_ALERT_ZONES:

    remaining = (
        alert_count
        - MAX_ALERT_ZONES
    )


    message_lines.append(
        f"+ {remaining} additional "
        f"alert zone(s)"
    )


message_lines.extend(
    [
        "",
        "Public-data situational-awareness prototype."
    ]
)


sms_body = "\n".join(
    message_lines
)


print(
    "SMS preview:"
)

print(
    sms_body
)


# ============================================================
# SEND REAL ALERTS
# ============================================================

success_count = 0
failure_count = 0


for subscriber in subscribers:

    to_number = (
        subscriber[
            "phone_number"
        ]
        .strip()
    )


    try:

        message = client.messages.create(
            body=sms_body,
            from_=from_number,
            to=to_number
        )


        success_count += 1


        print(
            "SMS sent successfully."
        )


        print(
            "Twilio SID:",
            message.sid
        )


        print(
            "Recipient ending:",
            to_number[-4:]
        )


    except Exception as err:

        failure_count += 1


        print(
            "SMS failed for recipient "
            f"ending {to_number[-4:]}"
        )


        print(
            "Twilio error:",
            str(err)
        )


# ============================================================
# SUMMARY
# ============================================================

print(
    "--------------------------------"
)


print(
    "Alert zones:",
    alert_count
)


print(
    "Subscribers:",
    subscriber_count
)


print(
    "SMS successful:",
    success_count
)


print(
    "SMS failed:",
    failure_count
)


# ============================================================
# EXIT STATUS
# ============================================================

if success_count > 0:

    dbutils.notebook.exit(
        f"ALERT_SENT_"
        f"{alert_count}_ZONES_"
        f"{success_count}_SUBSCRIBERS"
    )

else:

    dbutils.notebook.exit(
        "ALERT_SEND_FAILED"
    )